# Parameter Efficient Fine-tuning (PEFT) - LoRA

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

/home/arre/anaconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(torch.cuda.device_count(), "CUDA devices available:")
for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

1 CUDA devices available:
Device 0: NVIDIA GeForce RTX 3070


## Load in model, tokenizer and setup quantization config

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Quantization config
bits_and_bytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load in model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bits_and_bytes_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load in tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), e

## Setup LoRA config

In [4]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)


model = get_peft_model(model, lora_config)
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear

## Setup Dataset

In [5]:
data = load_dataset("openai/gsm8k", "main", split="train[:200]")
print(len(data))

200


In [6]:
def tokenize(batch):
    texts = [
        f"### Instruction:\n{instruction}\n### Reponse:\n{output}"
        for instruction, output in zip(batch["question"], batch["answer"])
    ]

    tokens = tokenizer(
        texts,
        padding = "max_length",
        truncation = True,
        max_length = 256,
        return_tensors = "pt"
    )

    tokens["labels"] = tokens["input_ids"].clone()

    return tokens

In [7]:
tokenized_data = data.map(tokenize, batched=True, remove_columns=data.column_names)

In [8]:
print(data[0])

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


## Setup training arguments

In [9]:
training_args = TrainingArguments(
    output_dir="./tinyllama_lora",
    per_device_train_batch_size=4,
    learning_rate=1e-4,
    num_train_epochs=10,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
)

## Start training

In [10]:
trainer.train()

Step,Training Loss
20,4.473900
40,1.821800
60,1.234800
80,1.073100
100,0.981000
120,0.899700
140,0.913300
160,0.882800
180,0.836100
200,0.836700


TrainOutput(global_step=500, training_loss=1.0208059101104736, metrics={'train_runtime': 143.4375, 'train_samples_per_second': 13.943, 'train_steps_per_second': 3.486, 'total_flos': 3181482344448000.0, 'train_loss': 1.0208059101104736, 'epoch': 10.0})

## Save model and tokenizer

In [12]:
model.save_pretrained("./tinyllama_lora_ft")
tokenizer.save_pretrained("./tinyllama_lora_ft")

('./tinyllama_lora_ft/tokenizer_config.json',
 './tinyllama_lora_ft/special_tokens_map.json',
 './tinyllama_lora_ft/tokenizer.json')